In [ ]:
# Jupyter notebook cells

# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.modulation import modulate, demodulate, get_constellation
from src.channel import awgn, multipath_fading, link_budget
from src.simulator import CommSimulator
from src.transmitter import create_frame
from src.receiver import receiver_chain

# Cell 2: Test Modulation
mod_type = '16QAM'
bits = np.random.randint(0, 2, 10000)
symbols = modulate(bits, mod_type)
rx_bits = demodulate(symbols, mod_type)
print(f"BER: {np.sum(bits != rx_bits) / len(bits)}")

# Plot constellation
const = get_constellation(mod_type)
plt.figure(figsize=(6,6))
plt.scatter(np.real(const), np.imag(const), s=100, color='red')
plt.grid(True)
plt.title(f'{mod_type} Constellation')
plt.axis('equal')
plt.show()

# Cell 3: Run Quick BER Simulation
sim = CommSimulator(mod_type='QPSK', sps=8)
snr, ber_sim, ber_theory = sim.run_ber_simulation(
    num_bits=20000,
    snr_range=np.arange(0, 15, 3),
    num_trials=3
)

plt.figure(figsize=(8,6))
plt.semilogy(snr, ber_sim, 'bo-', label='Simulated')
plt.semilogy(snr, ber_theory, 'r--', label='Theoretical')
plt.grid(True)
plt.xlabel('SNR (dB)')
plt.ylabel('BER')
plt.legend()
plt.show()

# Cell 4: Visualize Transmitted Signal
num_bits = 100
data_bits = np.random.randint(0, 2, num_bits)
tx_signal, symbols, preamble = create_frame(data_bits, 'QPSK', sps=8)

plt.figure(figsize=(12,4))
plt.plot(np.real(tx_signal[:500]))
plt.title('Transmitted Signal (I component)')
plt.grid(True)
plt.xlabel('Sample')
plt.show()

# Cell 5: Test Multipath
rx_signal = awgn(tx_signal, 10)
rx_signal_mp = multipath_fading(tx_signal)

plt.figure(figsize=(12,4))
plt.plot(np.real(rx_signal[:500]), label='AWGN only')
plt.plot(np.real(rx_signal_mp[:500]), label='Multipath + AWGN', alpha=0.7)
plt.legend()
plt.grid(True)
plt.title('Channel Effects Comparison')
plt.show()

# Cell 6: Complete System Test
snr_db = 8
rx_signal = awgn(tx_signal, snr_db)
rx_bits, H, sym = receiver_chain(rx_signal, preamble, 'QPSK', sps=8)

errors = np.sum(data_bits != rx_bits[:len(data_bits)])
ber = errors / len(data_bits)
print(f"SNR = {snr_db} dB, BER = {ber:.4f}")
print(f"Channel estimate: {H:.2f}")

# Cell 7: Link Budget Example
freq = 2.4e9  # 2.4 GHz
distances = np.linspace(100, 10000, 50)
rx_power = []
for d in distances:
    p, _ = link_budget(freq=freq, distance=d, tx_power=20)
    rx_power.append(p)

plt.figure(figsize=(8,6))
plt.plot(distances/1000, rx_power)
plt.grid(True)
plt.xlabel('Distance (km)')
plt.ylabel('Received Power (dBm)')
plt.title('Link Budget: Received Power vs Distance')
plt.show()